### Defining Hardware Lookup Table
Since we want to optimize over the latency and VRAM of our model, we'll create a lookup table.  This will record the latency and VRAM of the different building blocks in our supernet model, so we can quickly estimate the hardware performance of a specific path.

In [ ]:
import torch
import time
import json

def generate_hardware_lut(supernet, search_space):
    supernet.eval().cuda()
    lut = {}  # dictionary where we'll store observed performance
    
    # iterate through all searchable dimensions
    for res in search_space['res']:
        for width in [0.65, 0.8, 1.0, 1.2]:
            for depth in search_space['depth']:
                for exp in search_space['exp']:
                    config = {
                        'res': res, 'width': width, 
                        'depth': [depth]*4, 'exp': [exp]*4
                    }
                    
                    # Measurement logic
                    dummy_input = torch.randn(1, 1, 224, 224).cuda() # dummy data
                    
                    # warm up gpu
                    for _ in range(10): _ = supernet(dummy_input, config)
                    
                    torch.cuda.synchronize()
                    start = time.time()
                    for _ in range(50): _ = supernet(dummy_input, config)
                    torch.cuda.synchronize()
                    
                    latency = (time.time() - start) / 50 * 1000 # ms
                    vram = torch.cuda.max_memory_allocated() / (1024**2) # MB
                    
                    key = f"r{res}_w{width}_d{depth}_e{exp}"
                    lut[key] = {'latency': latency, 'vram': vram}
                    
    with open("hardware_lut.json", "w") as f:
        json.dump(lut, f)
    return lut

### Evolutionary Search
From our trained supernet, we want to "evolve" the best student architecture.  We will look to balance accuracy and latency, all within our hardware constraints (which is easy to check thanks to our lookup table).

In [ ]:
from train_loop import sample_configs
import random

# function to grab accuracy score
def evaluate_accuracy(supernet, config, val_loader):
    supernet.eval()
    correct = 0
    total = 0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
        
            outputs = supernet(images, config)    
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    return accuracy


# function for breeding the next generation
def create_next_generation(parents, population_size=50, mutation_rate=0.2):
    next_gen = []
    
    # keep parents
    for p in parents:
        next_gen.append(p['config'])
        
    while len(next_gen) < population_size:
        # pick two random parents
        p1 = random.choice(parents)['config']
        p2 = random.choice(parents)['config']
        
        # CROSSOVER - mix traits from both parents
        child = {
            'res': random.choice([p1['res'], p2['res']]),
            'width': random.choice([p1['width'], p2['width']]),
            # swap depth/exp stages
            'depth': [p1['depth'][i] if random.random() > 0.5 else p2['depth'][i] for i in range(4)],
            'exp': [p1['exp'][i] if random.random() > 0.5 else p2['exp'][i] for i in range(4)]
        }
        
        # MUTATE - occasionally flip a trait to a totally random value to explore
        if random.random() < mutation_rate:
            mutation_type = random.choice(['res', 'width', 'depth', 'exp'])
            if mutation_type == 'res':
                child['res'] = random.choice([128, 160, 190, 224])
            elif mutation_type == 'width':
                child['width'] = random.choice([0.65, 0.8, 1.0, 1.2])
            elif mutation_type == 'depth':
                idx = random.randint(0, 3)
                child['depth'][idx] = random.randint(2, 4)
            elif mutation_type == 'exp':
                idx = random.randint(0, 3)
                child['exp'][idx] = random.choice([3, 4, 6])
                
        next_gen.append(child)
        
    return next_gen


In [ ]:
# function to orchestrate evolutionary search
def evolutionary_search(supernet, val_loader, lut, generations=20, population_size=50):
    # initialize random population of configs
    population = sample_configs(population_size)
    pareto_front = []

    for gen in range(generations):
        fitness_scores = []
        
        for config in population:
            # first check hardware estimation from lookup table
            hw_key = f"r{config['res']}_w{config['width']}_d{config['depth'][0]}_e{config['exp'][0]}"
            if lut[hw_key]['vram'] > 5500: # 5.5GB limit
                continue
                
            # evaluate accuracy on validation set
            acc = evaluate_accuracy(supernet, config, val_loader)
            latency = lut[hw_key]['latency']
            
            fitness_scores.append({'config': config, 'acc': acc, 'lat': latency})

        # sort by accuracy and latency
        # keep the top 20% and mutate
        fitness_scores.sort(key=lambda x: x['acc'], reverse=True)
        parents = fitness_scores[:10]
        pareto_front.extend(parents)
        
        # crossover & mutation to create next generation
        population = create_next_generation(parents)
        print(f"Gen {gen} Best Acc: {parents[0]['acc']:.2f}%")  # report metrics

    return pareto_front